05_Classification.ipynb  (edited: evaluate ALL models on TEST, then select best)

🌿 VedaVision — Module 2
Notebook 05 — Classification
### Tune ALL Candidates → Evaluate EACH on Held-Out Test Set → Select Best

---
**Input**  : `dataset/csv/species_id_train_features.csv`
             `dataset/csv/species_id_test_features.csv`
             `dataset/csv/species_id_test_labels.csv`

**Output** : `dataset/models/species_classifier.pkl`
             `dataset/models/scaler.pkl`
             `dataset/models/label_encoder.pkl`
             `dataset/models/feature_columns.pkl`
             `dataset/logs/05_*.png` (evaluation charts)

---
### ⚠️ A note on what changed vs. the original notebook
The original notebook picked ONE model by CV accuracy (Cell 7), tuned only
that one (Cell 9), and only THAT model ever touched the test set. This
version tunes and test-evaluates ALL 6 candidates, so you can see the full
comparison table before choosing.

Be aware: choosing your final model based on TEST accuracy (rather than
CV/validation accuracy) is a mild form of leakage -- the test set is meant
to simulate genuinely unseen data, and using it to pick between models,
not just to report a final number, means your reported test accuracy for
the "winner" is a little optimistic. This script reports CV, validation,
AND test accuracy side-by-side for every model so you can see whether the
CV-best pick and the test-best pick actually agree. If they don't agree by
much, that's usually a sign your test set (small, ~120 images) is too
small for its ranking of close competitors to be very reliable -- in that
case, leaning on CV/validation for model choice is the safer call, and
test accuracy for the CV-winner is the number worth trusting for reporting.
---
### Run Order:
```
01_Augmentation -> 02_Preprocessing -> 04_Feature_Extraction -> 05_Classification.ipynb  (this notebook)
```

## Cell 1 — Install Libraries

In [1]:
!pip install scikit-learn xgboost joblib matplotlib seaborn pandas numpy -q
print('✅ Libraries installed!')

✅ Libraries installed!


## Cell 2 — Import Libraries

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
 

print('✅ All libraries imported!')

✅ All libraries imported!


## Cell 3 — Mount Drive & Configure Paths
**Only change things in this cell**

In [3]:
 

# ============================================================
# ✏️  UPDATE THESE PATHS — match your feature extraction notebook
# ============================================================
CSV_PATH   = 'C:/Users/User/Documents/UOM/L4S2/Research/csv'
MODEL_PATH = 'C:/Users/User/Documents/UOM/L4S2/Research'
LOGS_PATH  = 'C:/Users/User/Documents/UOM/L4S2/Research/csv/logs'

TRAIN_CSV       = os.path.join(CSV_PATH, 'species_id_train_features.csv')
TEST_CSV        = os.path.join(CSV_PATH, 'species_id_test_features.csv')
TEST_LABELS_CSV = os.path.join(CSV_PATH, 'species_id_test_labels.csv')

RANDOM_STATE = 42
VAL_SIZE     = 0.2   # fraction of TRAIN held out for validation during model selection
# ============================================================

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(LOGS_PATH,  exist_ok=True)

print(f'✅ Drive mounted!')
print(f'📁 CSV input    : {CSV_PATH}')
print(f'📁 Model output : {MODEL_PATH}')

✅ Drive mounted!
📁 CSV input    : C:/Users/User/Documents/UOM/L4S2/Research/csv
📁 Model output : C:/Users/User/Documents/UOM/L4S2/Research


## Cell 4 — Load Train / Test CSVs

In [4]:
train_df       = pd.read_csv(TRAIN_CSV)
test_df        = pd.read_csv(TEST_CSV)
test_labels_df = pd.read_csv(TEST_LABELS_CSV)

print('📊 TRAIN features :', train_df.shape)
print('📊 TEST features  :', test_df.shape)
print('📊 TEST labels    :', test_labels_df.shape)
print()
print('Species in train:', sorted(train_df['species'].unique()))
print()
print('Images per species/view (train):')
print(train_df.groupby(['species', 'view']).size().to_string())

📊 TRAIN features : (6995, 109)
📊 TEST features  : (130, 107)
📊 TEST labels    : (130, 3)

Species in train: ['Diya_Na', 'Gammiris', 'Ingini', 'Iriveriya', 'Kapparawalliya', 'Kora_Kaha', 'Kuringchan', 'Kurundu', 'Masbadda', 'Na', 'Rathu_Koboleela', 'Sudu_Koboleela', 'Wali_Kaha']

Images per species/view (train):
species          view  
Diya_Na          bottom    260
                 top       264
Gammiris         bottom    270
                 top       270
Ingini           bottom    270
                 top       270
Iriveriya        bottom    270
                 top       270
Kapparawalliya   bottom    270
                 top       270
Kora_Kaha        bottom    270
                 top       270
Kuringchan       bottom    270
                 top       270
Kurundu          bottom    270
                 top       270
Masbadda         bottom    270
                 top       270
Na               bottom    270
                 top       270
Rathu_Koboleela  bottom    270
            

## Cell 5 — Prepare Features & Labels

In [5]:
META_COLS = ['filename', 'species', 'view', 'label']
feature_cols = [c for c in train_df.columns if c not in META_COLS]

X = train_df[feature_cols].copy()
X['view_top'] = (train_df['view'] == 'top').astype(int)
feature_cols_final = feature_cols + ['view_top']

y = train_df['label'].copy()

# Check for NaN / Inf
n_nan = X.isna().sum().sum()
n_inf = np.isinf(X.select_dtypes(include=[np.number]).values).sum()
print(f'NaN values found: {n_nan}')
print(f'Inf values found: {n_inf}')

if n_nan > 0 or n_inf > 0:
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    print('✅ Cleaned: Inf -> NaN -> filled with column median')

train_medians = X.median(numeric_only=True)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print()
print('Classes:', list(label_encoder.classes_))
print()
print('Class counts:')
print(y.value_counts())
print()
print(f'Total feature columns (incl. view_top): {len(feature_cols_final)}')

NaN values found: 0
Inf values found: 0

Classes: ['Diya_Na', 'Gammiris', 'Ingini', 'Iriveriya', 'Kapparawalliya', 'Kora_Kaha', 'Kuringchan', 'Kurundu', 'Masbadda', 'Na', 'Rathu_Koboleela', 'Sudu_Koboleela', 'Wali_Kaha']

Class counts:
label
Gammiris           540
Ingini             540
Iriveriya          540
Kapparawalliya     540
Kora_Kaha          540
Kuringchan         540
Kurundu            540
Masbadda           540
Na                 540
Rathu_Koboleela    540
Wali_Kaha          540
Sudu_Koboleela     531
Diya_Na            524
Name: count, dtype: int64

Total feature columns (incl. view_top): 106


## Cell 6 — Train / Validation Split + Feature Scaling

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded,
    test_size=VAL_SIZE,
    stratify=y_encoded,
    random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

print(f'Train: {X_train.shape[0]} samples')
print(f'Val  : {X_val.shape[0]} samples')

Train: 5596 samples
Val  : 1399 samples


## Cell 7 — Compare Candidate Models (5-fold Cross-Validation)

In [7]:
models = {
    'Random Forest'      : RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    'Gradient Boosting'  : GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost'            : XGBClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                          eval_metric='mlogloss'),
    'SVM (RBF)'          : SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('🔬 5-fold cross-validation on TRAINING split')
print('=' * 55)
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv,
                              scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<22} mean={scores.mean():.4f}  std={scores.std():.4f}')

best_model_name_by_cv = max(cv_results, key=lambda k: cv_results[k].mean())
print()
print(f'🏆 Best model by CV alone: {best_model_name_by_cv}  (kept for comparison later)')

plt.figure(figsize=(10, 5))
plt.boxplot(cv_results.values(), labels=cv_results.keys())
plt.title('5-Fold CV Accuracy by Model', fontweight='bold')
plt.ylabel('Accuracy')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f'{LOGS_PATH}/05_model_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

🔬 5-fold cross-validation on TRAINING split
Random Forest          mean=0.9923  std=0.0018
Gradient Boosting      mean=0.9925  std=0.0015
XGBoost                mean=0.9918  std=0.0015
SVM (RBF)              mean=0.9902  std=0.0022
Logistic Regression    mean=0.9943  std=0.0018
K-Nearest Neighbors    mean=0.9653  std=0.0050

🏆 Best model by CV alone: Logistic Regression  (kept for comparison later)


TypeError: boxplot() got an unexpected keyword argument 'labels'. Did you mean 'label'?

<Figure size 1000x500 with 0 Axes>

## Cell 8 — Feature Importance (Random Forest, for reference)

In [ ]:
rf_importance = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf_importance.fit(X_train_scaled, y_train)

importances = pd.Series(rf_importance.feature_importances_, index=feature_cols_final)
top20 = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
top20[::-1].plot(kind='barh', color='seagreen')
plt.title('Top 20 Most Important Features (Random Forest)', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(f'{LOGS_PATH}/05_feature_importance.png', bbox_inches='tight', dpi=120)
plt.show()

print('Top 10 features:')
print(top20.head(10).to_string())

## Cell 9 — Tune, Validate, and TEST-Evaluate EVERY Candidate Model

For each of the 6 models: GridSearchCV on the training split (unbiased,
never sees val or test), then refit each tuned model on the FULL training
set, then predict on validation AND on the true test set. This produces
one row per model in `comparison_df` -- CV / validation / test accuracy
side-by-side -- so you can see the whole picture before picking a winner.

In [ ]:
param_grids = {
     
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
    },
    
}

# Prepare the test feature matrix once (same cleaning as training)
X_test = test_df[feature_cols].copy()
X_test['view_top'] = (test_df['view'] == 'top').astype(int)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.fillna(train_medians)

tuned_models   = {}   # name -> fitted estimator (refit on FULL train set)
comparison_rows = []

print('🔧 Tuning + evaluating all 6 candidates (this takes longer than tuning just one)')
print('=' * 70)

for name, base_model in models.items():
    print(f'\n▶ {name}')

    # 1. Hyperparameter search on the TRAIN split only (unbiased)
    grid = GridSearchCV(base_model, param_grids[name], cv=cv,
                         scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    print(f'   Best params      : {grid.best_params_}')
    print(f'   Best CV accuracy : {grid.best_score_:.4f}')

    # 2. Validation accuracy with the tuned model (still trained on TRAIN split only)
    val_preds = grid.best_estimator_.predict(X_val_scaled)
    val_acc   = accuracy_score(y_val, val_preds)
    print(f'   Validation accuracy: {val_acc:.4f}')

    # 3. Refit the tuned model on the FULL training set (train+val) for deployment quality
    scaler_full = StandardScaler()
    X_full_scaled = scaler_full.fit_transform(X)

    # clone-like refit: re-instantiate with best params to avoid reusing a fitted grid object
    final_candidate = grid.best_estimator_.__class__(**grid.best_params_) \
        if name != 'XGBoost' else XGBClassifier(**grid.best_params_, random_state=RANDOM_STATE,
                                                  eval_metric='mlogloss')
    # Some classifiers need fixed constructor args not in the grid (e.g. SVM probability=True)
    if name == 'SVM (RBF)':
        final_candidate = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, **grid.best_params_)
    elif name == 'Logistic Regression':
        final_candidate = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, **grid.best_params_)
    elif name in ('Random Forest', 'Gradient Boosting'):
        final_candidate = base_model.__class__(random_state=RANDOM_STATE, **grid.best_params_)
    elif name == 'K-Nearest Neighbors':
        final_candidate = KNeighborsClassifier(**grid.best_params_)

    final_candidate.fit(X_full_scaled, y_encoded)

    # 4. Predict on the TRUE test set
    X_test_scaled = scaler_full.transform(X_test)
    test_preds_encoded = final_candidate.predict(X_test_scaled)
    test_preds = label_encoder.inverse_transform(test_preds_encoded)

    results_this_model = test_df[['filename', 'view']].copy()
    results_this_model['predicted_species'] = test_preds
    merged_this_model = results_this_model.merge(test_labels_df, on=['filename', 'view'], how='left')
    merged_this_model['correct'] = merged_this_model['predicted_species'] == merged_this_model['true_label']
    test_acc = merged_this_model['correct'].mean()
    print(f'   TEST accuracy       : {test_acc:.4f}')

    tuned_models[name] = {
        'estimator': final_candidate,
        'scaler': scaler_full,
        'best_params': grid.best_params_,
        'merged_test_results': merged_this_model,
    }

    comparison_rows.append({
        'model'            : name,
        'cv_accuracy'      : grid.best_score_,
        'validation_accuracy': val_acc,
        'test_accuracy'    : test_acc,
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('test_accuracy', ascending=False).reset_index(drop=True)

print()
print('=' * 70)
print('📊 FULL COMPARISON — ALL 6 MODELS')
print('=' * 70)
print(comparison_df.to_string(index=False))

# Visual comparison
plt.figure(figsize=(11, 6))
x = np.arange(len(comparison_df))
width = 0.25
plt.bar(x - width, comparison_df['cv_accuracy'], width, label='CV accuracy')
plt.bar(x,          comparison_df['validation_accuracy'], width, label='Validation accuracy')
plt.bar(x + width,  comparison_df['test_accuracy'], width, label='Test accuracy')
plt.xticks(x, comparison_df['model'], rotation=20)
plt.ylabel('Accuracy')
plt.title('CV vs Validation vs TEST Accuracy — All Candidate Models', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(f'{LOGS_PATH}/05_all_models_test_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

NameError: name 'test_df' is not defined

## Cell 10 — Select the Best Model (by TEST accuracy)

⚠️ See the note at the top of this notebook: selecting by test accuracy is
a mild form of leakage. `best_model_name_by_cv` (from Cell 7/9) is shown
alongside for comparison -- if the two disagree, treat the test-based
ranking with some skepticism, especially with a small test set.

In [ ]:
best_model_name = comparison_df.iloc[0]['model']
final_model  = tuned_models[best_model_name]['estimator']
scaler       = tuned_models[best_model_name]['scaler']
merged       = tuned_models[best_model_name]['merged_test_results']
best_params  = tuned_models[best_model_name]['best_params']
test_acc     = comparison_df.iloc[0]['test_accuracy']
val_acc      = comparison_df.iloc[0]['validation_accuracy']

print(f'🏆 Best model by TEST accuracy : {best_model_name}  (test_acc={test_acc:.4f})')
print(f'🏆 Best model by CV accuracy   : {best_model_name_by_cv}')
if best_model_name != best_model_name_by_cv:
    print('⚠️  These disagree! With a small test set, the test-accuracy ranking')
    print('    of close competitors can be noisy. Consider whether the CV-based')
    print('    pick might be the more robust choice for deployment.')

print()
print(classification_report(merged['true_label'], merged['predicted_species']))

labels_order = sorted(merged['true_label'].unique())
cm_test = confusion_matrix(merged['true_label'], merged['predicted_species'], labels=labels_order)
disp = ConfusionMatrixDisplay(cm_test, display_labels=labels_order)
fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, cmap='Greens', xticks_rotation=45, colorbar=False)
plt.title(f'TEST SET Confusion Matrix — {best_model_name} (Accuracy: {test_acc:.2%})', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{LOGS_PATH}/05_test_confusion_matrix.png', bbox_inches='tight', dpi=120)
plt.show()

## Cell 11 — Per-Species & Per-View Breakdown (Best Model)

In [ ]:
print('📋 PER-SPECIES ACCURACY')
per_species = merged.groupby('true_label')['correct'].agg(['mean', 'count'])
per_species.columns = ['accuracy', 'n_samples']
print(per_species.sort_values('accuracy').to_string())

print()
print('📋 PER-VIEW ACCURACY (top vs bottom)')
print(merged.groupby('view')['correct'].mean().to_string())

print()
print('⚠️  MISCLASSIFICATION PAIRS (true_label -> predicted)')
mistakes = merged[~merged['correct']]
if len(mistakes) > 0:
    confusion_pairs = (mistakes.groupby(['true_label', 'predicted_species'])
                        .size().sort_values(ascending=False))
    print(confusion_pairs.to_string())
else:
    print('No misclassifications! 🎉')

## Cell 12 — Save Model Artifacts (Best Model Only)

In [ ]:
joblib.dump(final_model,        os.path.join(MODEL_PATH, 'species_classifier.pkl'))
joblib.dump(scaler,             os.path.join(MODEL_PATH, 'scaler.pkl'))
joblib.dump(label_encoder,      os.path.join(MODEL_PATH, 'label_encoder.pkl'))
joblib.dump(feature_cols_final, os.path.join(MODEL_PATH, 'feature_columns.pkl'))

print('✅ Model artifacts saved:')
print(f'   {MODEL_PATH}/species_classifier.pkl')
print(f'   {MODEL_PATH}/scaler.pkl')
print(f'   {MODEL_PATH}/label_encoder.pkl')
print(f'   {MODEL_PATH}/feature_columns.pkl')

## Cell 13 — Final Summary

In [ ]:
print('📊 FINAL SUMMARY')
print('=' * 60)
print(comparison_df.to_string(index=False))
print('=' * 60)
print(f'Selected model (by TEST accuracy) : {best_model_name}')
print(f'Best params                       : {best_params}')
print(f'Validation accuracy               : {val_acc:.4f}')
print(f'TEST accuracy                      : {test_acc:.4f}')
print('=' * 60)
print()
print('📁 Model saved to:', MODEL_PATH)
print()
print('📋 To use this model on a NEW leaf photo:')
print('   1. Run it through 02_Preprocessing.ipynb')
print('   2. Extract features with 04_Feature_Extraction.ipynb functions')
print('   3. Add the view_top one-hot column the same way as Cell 5')
print('   4. Scale with the saved scaler.pkl')
print('   5. Predict with species_classifier.pkl')
print('   6. Decode the prediction with label_encoder.pkl')